# Health Claim Checker — Exploratory Data Analysis

Dataset: `ImperialCollegeLondon/health_fact` (stored as parquet in `data/train/`)

Binary label mapping:
- 0 (false/misleading) → 1 MISINFORMATION
- 1 (true) → 0 RELIABLE
- 2 (mixture) → 1 MISINFORMATION
- 3 (unproven) → 1 MISINFORMATION
- -1 → dropped

In [ ]:
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

LABEL_MAP = {0: 1, 1: 0, 2: 1, 3: 1, -1: None}
BINARY_NAMES = {0: 'RELIABLE', 1: 'MISINFORMATION'}
ORIG_NAMES = {0: 'false', 1: 'true', 2: 'mixture', 3: 'unproven', -1: 'invalid'}

files = sorted(glob.glob('data/train/*.parquet'))
df_raw = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

df = df_raw.copy()
df['label_bin'] = df['label'].map(LABEL_MAP)
df = df.dropna(subset=['label_bin'])
df['label_bin'] = df['label_bin'].astype(int)
df['label_name'] = df['label_bin'].map(BINARY_NAMES)
df['orig_label_name'] = df['label'].map(ORIG_NAMES)

print(f'Total examples (after dropping -1): {len(df)}')
print(f'Columns: {df.columns.tolist()}')

## Label distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts_bin = df['label_name'].value_counts()
axes[0].bar(counts_bin.index, counts_bin.values, color=['steelblue', 'tomato'])
axes[0].set_title('Binary Label Distribution')
axes[0].set_ylabel('Count')
for i, (name, val) in enumerate(counts_bin.items()):
    axes[0].text(i, val + 5, str(val), ha='center')

counts_orig = df['orig_label_name'].value_counts()
axes[1].bar(counts_orig.index, counts_orig.values, color='slategray')
axes[1].set_title('Original Label Distribution')
axes[1].set_ylabel('Count')
for i, (name, val) in enumerate(counts_orig.items()):
    axes[1].text(i, val + 2, str(val), ha='center')

plt.tight_layout()
plt.show()
print(f'Class imbalance ratio: {counts_bin.max() / counts_bin.min():.1f}x')

## Claim length analysis

In [ ]:
df['claim_words'] = df['claim'].str.split().str.len()
df['has_main_text'] = df['main_text'].notna() & (df['main_text'].str.strip() != '')

print('Claim word count stats:')
print(df['claim_words'].describe().round(1))
print(f'\nExamples with main_text: {df["has_main_text"].sum()} / {len(df)} ({df["has_main_text"].mean()*100:.1f}%)')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for label, grp in df.groupby('label_name'):
    color = 'steelblue' if label == 'RELIABLE' else 'tomato'
    axes[0].hist(grp['claim_words'], bins=30, alpha=0.6, label=label, color=color)
axes[0].set_title('Claim word count by label')
axes[0].set_xlabel('Words')
axes[0].legend()

df.groupby('label_name')['has_main_text'].mean().plot.bar(ax=axes[1], color=['steelblue', 'tomato'])
axes[1].set_title('Fraction with main_text article by label')
axes[1].set_ylabel('Fraction')
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()

## Sample claims

In [ ]:
print('=== RELIABLE examples ===')
for _, row in df[df['label_bin'] == 0].sample(3, random_state=1).iterrows():
    print(f'  Claim: {row["claim"]}')
    print(f'  Explanation: {str(row["explanation"])[:150]}...')
    print()

print('=== MISINFORMATION examples ===')
for _, row in df[df['label_bin'] == 1].sample(3, random_state=1).iterrows():
    print(f'  Claim: {row["claim"]}')
    print(f'  Explanation: {str(row["explanation"])[:150]}...')
    print()

## Quick baseline: TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

def make_text(row):
    claim = str(row.get('claim') or '').strip()
    mt = row.get('main_text')
    if pd.isna(mt) or not str(mt).strip():
        return claim
    return f"{claim}\n\n{str(mt)[:2000]}"

texts = [make_text(row) for _, row in df.iterrows()]
y = df['label_bin'].to_numpy()

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=20000, ngram_range=(1, 2), min_df=2)),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced')),
])

scores = cross_val_score(pipe, texts, y, cv=5, scoring='f1', n_jobs=-1)
print(f'TF-IDF + LR  5-fold F1:  {scores.mean():.3f} +/- {scores.std():.3f}')
scores_acc = cross_val_score(pipe, texts, y, cv=5, scoring='accuracy', n_jobs=-1)
print(f'TF-IDF + LR  5-fold Acc: {scores_acc.mean():.3f} +/- {scores_acc.std():.3f}')

## Explanation length distribution

In [ ]:
df['expl_words'] = df['explanation'].astype(str).str.split().str.len()
print('Explanation word count stats:')
print(df['expl_words'].describe().round(1))

fig, ax = plt.subplots(figsize=(8, 3))
for label, grp in df.groupby('label_name'):
    color = 'steelblue' if label == 'RELIABLE' else 'tomato'
    ax.hist(grp['expl_words'].clip(upper=500), bins=30, alpha=0.6, label=label, color=color)
ax.set_title('Explanation length (words, clipped at 500)')
ax.set_xlabel('Words')
ax.legend()
plt.tight_layout()
plt.show()